# Words and Embeddings

## Why Can't Computers Just Read Words?

A computer can store the string `"dog"` — but it has no idea that **dog** and **puppy** are related, that **hot** is the opposite of **cold**, or that **Paris** is to **France** what **Tokyo** is to **Japan**.

To work with meaning, we need to convert words into **numbers** — specifically, into vectors that live in a high-dimensional space where **similar words end up close together**.

This idea is called a **word embedding**, and it is the foundation of every modern language model.

---
## 1. The Old Way — Bag of Words

Before embeddings, NLP used a simple trick: represent each document as a count of how many times each word appears.

**Example:**

| Sentence | cat | sat | mat | dog | ran |
|----------|-----|-----|-----|-----|-----|
| "the cat sat on the mat" | 1 | 1 | 1 | 0 | 0 |
| "the dog ran on the mat" | 0 | 0 | 1 | 1 | 1 |

This works for some tasks. But it has a fatal flaw — **word order is completely lost**.

- *"The dog bit the man"*
- *"The man bit the dog"*

Same bag. Completely different meaning.

And the vectors tell us nothing about relationships — `cat` and `dog` look just as different as `cat` and `skyscraper`.

In [1]:
from collections import Counter

sentences = [
    "the cat sat on the mat",
    "the dog ran on the mat",
    "the man bit the dog",
    "the dog bit the man"
]

# Collect every word from every sentence
all_words = []
for sentence in sentences:
    words = sentence.split()
    for word in words:
        all_words.append(word)

# Remove duplicates and sort alphabetically
vocab = sorted(set(all_words))
# vocab = sorted(set(w for s in sentences for w in s.split()))
# ~1.5x faster — bytecode-optimized LIST_APPEND vs repeated .append() method lookup

print("Vocabulary:", vocab)

# Build a bag-of-words vector for each sentence
bow_vectors = []
for sentence in sentences:
    counts = Counter(sentence.split())
    vector = []
    for word in vocab:
        vector.append(counts.get(word, 0))
    # vector = [counts.get(word, 0) for word in vocab]
    # ~1.5x faster at scale — list comprehension avoids repeated .append() overhead
    bow_vectors.append(vector)
    print(f"\n'{sentence}'")
    print(f"  vector: {vector}")

Vocabulary: ['bit', 'cat', 'dog', 'man', 'mat', 'on', 'ran', 'sat', 'the']

'the cat sat on the mat'
  vector: [0, 1, 0, 0, 1, 1, 0, 1, 2]

'the dog ran on the mat'
  vector: [0, 0, 1, 0, 1, 1, 1, 0, 2]

'the man bit the dog'
  vector: [1, 0, 1, 1, 0, 0, 0, 0, 2]

'the dog bit the man'
  vector: [1, 0, 1, 1, 0, 0, 0, 0, 2]


---
## 2. A Better Counter — TF-IDF

**TF-IDF** (Term Frequency - Inverse Document Frequency) improves on Bag of Words by weighting words by how *uniquely* important they are to a document.

**Abbreviation full forms:**
- **TF** — Term Frequency: how often a word appears in *this* document
- **IDF** — Inverse Document Frequency: how rare the word is across *all* documents
- **TF-IDF** — Term Frequency - Inverse Document Frequency

**Intuition:**
- A word that appears often in one document but rarely in others → probably important → high score
- A word like "the" that appears everywhere → not useful → low score

**Formula:**


TF(word, doc) = $\frac{count(word\ in\ doc)}{total\ words\ in\ doc}$

IDF(word) = $\log{\frac{total\ docs}{docs\ containing\ word}}$

TF-IDF = TF × IDF


**Still has the core problem:** "dog" and "puppy" still look completely unrelated — they're just two separate columns with no connection between them.

In [ ]:
import math

def tf(word, document):
    words = document.split()
    return words.count(word) / len(words)

def idf(word, documents):
    count = 0
    for doc in documents:
        if word in doc.split():
            count += 1
    # count = sum(1 for doc in documents if word in doc.split())
    # ~2x faster at scale — generator expression avoids building intermediate list
    return math.log(len(documents) / (1 + count))

def tfidf(word, document, documents):
    return tf(word, document) * idf(word, documents)

docs = [
    "the cat sat on the mat",
    "the dog ran on the mat",
    "cats and dogs are both animals",
    "the mat is on the floor"
]

# Compare TF-IDF scores for a few words in doc[0]
target_doc = docs[0]
words_to_check = ["cat", "mat", "the", "sat"]

print(f"Document: '{target_doc}'\n")
print(f"{'Word':<10} {'TF':>8} {'IDF':>8} {'TF-IDF':>10}")
print("-" * 40)
for word in words_to_check:
    t = tf(word, target_doc)
    i = idf(word, docs)
    ti = tfidf(word, target_doc, docs)
    print(f"{word:<10} {t:>8.3f} {i:>8.3f} {ti:>10.4f}")

print("\nNotice: 'the' scores very low — it appears in all docs.")
print("'cat' scores high — it appears only in this doc.")

---
## 3. The Modern Way — Word Embeddings

The key insight: instead of representing a word as a sparse count vector (mostly zeros), represent it as a **dense vector of learned numbers**.

```
"dog"  →  [0.23, -0.81, 0.44, 0.12, ...]   # 300 numbers
"cat"  →  [0.21, -0.79, 0.41, 0.15, ...]   # 300 numbers
"bank" →  [0.05,  0.33, -0.12, 0.88, ...]  # 300 numbers
```

These numbers are not hand-crafted — they are **learned from data**. The model reads billions of sentences and adjusts vectors so that:

- Words that appear in similar contexts → end up with similar vectors
- Words that appear in different contexts → end up with different vectors

This is called the **distributional hypothesis**:
> "A word is known by the company it keeps." — J.R. Firth, 1957

**Dog and cat** both appear near words like *"pet"*, *"food"*, *"veterinarian"* → their vectors end up close.  
**Dog and skyscraper** never share context → their vectors end up far apart.

In [ ]:
import numpy as np

# Simulate simplified word vectors (in reality these are 100-300 dimensions)
# These are hand-crafted to illustrate relationships — real vectors are learned
np.random.seed(42)

# We'll create 4D toy vectors that encode two properties:
# Dimension 0-1: "animal-ness" and "size"
# Dimension 2-3: "royalty" and "gender"

word_vectors = {
    "dog":        np.array([0.9,  0.4,  0.0,  0.0]),
    "cat":        np.array([0.9,  0.2,  0.0,  0.0]),
    "puppy":      np.array([0.9,  0.1,  0.0,  0.0]),
    "wolf":       np.array([0.9,  0.7,  0.0,  0.0]),
    "skyscraper": np.array([0.0,  1.0,  0.0,  0.0]),
    "king":       np.array([0.0,  0.5,  0.9,  0.8]),
    "queen":      np.array([0.0,  0.5,  0.9,  0.2]),
    "man":        np.array([0.0,  0.5,  0.0,  0.8]),
    "woman":      np.array([0.0,  0.5,  0.0,  0.2]),
}

def cosine_similarity(vector_a, vector_b):
    dot_product = np.dot(vector_a, vector_b)
    magnitude_a = np.linalg.norm(vector_a)
    magnitude_b = np.linalg.norm(vector_b)
    return dot_product / (magnitude_a * magnitude_b)

# Compare similarities
pairs = [
    ("dog", "cat"),
    ("dog", "puppy"),
    ("dog", "skyscraper"),
    ("king", "queen"),
    ("king", "man"),
]

print(f"{'Word 1':<12} {'Word 2':<12} {'Cosine Similarity':>18}")
print("-" * 45)
for w1, w2 in pairs:
    sim = cosine_similarity(word_vectors[w1], word_vectors[w2])
    print(f"{w1:<12} {w2:<12} {sim:>18.4f}")

---
## 4. Vector Arithmetic — Meaning Has Geometry

One of the most famous discoveries about word embeddings: **you can do arithmetic on meaning**.

```
vector("king") - vector("man") + vector("woman") ≈ vector("queen")
```

This works because the embedding space encodes **relationships** as directions:

```
"man" → "woman"  is the same direction as  "king" → "queen"
```

The vector that points from man to woman (the "gender direction") is the same vector that points from king to queen.

This is not programmed in — it **emerges** from the training data.

In [ ]:
# Demonstrate vector arithmetic with our toy vectors

def find_closest(target_vector, word_vectors, exclude=None):
    exclude = exclude or []
    best_word = None
    best_sim = -1
    for word, vector in word_vectors.items():
        if word in exclude:
            continue
        sim = cosine_similarity(target_vector, vector)
        if sim > best_sim:
            best_sim = sim
            best_word = word
    return best_word, best_sim

# king - man + woman = ?
result_vector = word_vectors["king"] - word_vectors["man"] + word_vectors["woman"]
closest_word, similarity = find_closest(result_vector, word_vectors, exclude=["king", "man", "woman"])

print("Vector arithmetic: king - man + woman")
print(f"  Result vector:    {result_vector}")
print(f"  Closest word:     '{closest_word}' (similarity: {similarity:.4f})")
print()

# dog - puppy relationship
result_vector = word_vectors["wolf"] - word_vectors["dog"] + word_vectors["puppy"]
closest_word, similarity = find_closest(result_vector, word_vectors, exclude=["wolf", "dog", "puppy"])
print("Vector arithmetic: wolf - dog + puppy")
print(f"  Closest word:     '{closest_word}' (similarity: {similarity:.4f})")

---
## 5. How Embeddings Are Learned — Word2Vec

**Word2Vec** is the algorithm that popularized word embeddings. It learns vectors entirely from raw text — no labels needed.

It uses a simple but clever training task:

**Skip-gram training task:**
> Given a center word, predict the surrounding context words.

```
Sentence: "the quick brown fox jumps over the lazy dog"

Center word: "fox"
Context (window=2): ["quick", "brown", "jumps", "over"]

Training signal:
  fox → quick  (should be high probability)
  fox → brown  (should be high probability)
  fox → king   (should be low probability)  ← not in context
```

**How it updates vectors:**
- Prediction correct → vectors stay similar
- Prediction wrong → adjust vectors slightly to improve

After millions of sentences and billions of updates, the vectors **settle** into positions that reflect real meaning.

**Two training modes:**

| Mode | Task | Strength |
|------|------|----------|
| Skip-gram | Center word → predict context | Works well for rare words |
| CBOW (Continuous Bag of Words) | Context words → predict center | Faster, works well for frequent words |

**Abbreviation full forms:**
- **Word2Vec** — Word to Vector
- **CBOW** — Continuous Bag of Words

In [ ]:
# Install gensim for real Word2Vec
# pip install gensim

# Demonstrate the training concept manually (simplified)
import numpy as np

np.random.seed(0)

# Toy corpus
corpus = [
    "the cat sat on the mat",
    "the dog sat on the mat",
    "cats and dogs are pets",
    "dogs are loyal animals",
    "cats are independent animals",
    "the puppy played with the cat",
    "the kitten played with the dog",
]

# Build vocabulary
all_words = []
for sentence in corpus:
    words = sentence.split()
    for word in words:
        all_words.append(word)
# all_words = [w for s in corpus for w in s.split()]
# ~20x faster at scale — list comprehension avoids repeated .append() overhead

vocab = list(set(all_words))

word_to_idx = {}
for idx, word in enumerate(vocab):
    word_to_idx[word] = idx
# word_to_idx = {word: idx for idx, word in enumerate(vocab)}
# ~5x faster at scale — dict comprehension avoids repeated dict assignment overhead

idx_to_word = {}
for word, idx in word_to_idx.items():
    idx_to_word[idx] = word
# idx_to_word = {idx: word for word, idx in word_to_idx.items()}
# ~5x faster at scale — same reason

print(f"Vocabulary size: {len(vocab)}")
print(f"Vocabulary: {sorted(vocab)}")

# Extract skip-gram pairs (center word, context word)
window_size = 2
skip_gram_pairs = []

for sentence in corpus:
    words = sentence.split()
    for center_idx, center_word in enumerate(words):
        context_start = max(0, center_idx - window_size)
        context_end = min(len(words), center_idx + window_size + 1)
        for context_idx in range(context_start, context_end):
            if context_idx != center_idx:
                skip_gram_pairs.append((center_word, words[context_idx]))

print(f"\nTotal skip-gram pairs: {len(skip_gram_pairs)}")
print("\nSample pairs (center → context):")
for pair in skip_gram_pairs[:10]:
    print(f"  '{pair[0]}' → '{pair[1]}'")

---
## 6. Using Real Pretrained Embeddings

Training Word2Vec from scratch requires large corpora. In practice, we load **pretrained embeddings** — vectors already trained on billions of words.

Popular pretrained embeddings:
- **Word2Vec** — trained on Google News (100 billion words)
- **GloVe** — trained on Wikipedia + Common Crawl
- **FastText** — handles subword information (understands "running" from "run")

**Abbreviation full forms:**
- **GloVe** — Global Vectors for Word Representation
- **FastText** — Fast Text (Facebook AI Research)

Below we use **gensim** to load GloVe vectors and explore real learned relationships.

In [ ]:
# Run this cell to install gensim if not already installed
import subprocess
subprocess.run(["pip", "install", "gensim", "--break-system-packages", "-q"])
print("gensim ready")

In [ ]:
import gensim.downloader as api

# Load GloVe 50-dimensional vectors (small, fast to download ~66MB)
print("Loading GloVe 50d vectors (first run downloads ~66MB)...")
glove = api.load("glove-wiki-gigaword-50")
print(f"Loaded! Vocabulary size: {len(glove):,} words")
print(f"Vector dimensions: {glove.vector_size}")

In [ ]:
# Explore real word similarities

print("=== Real Word Similarities ===\n")

word_pairs = [
    ("dog", "cat"),
    ("dog", "puppy"),
    ("dog", "skyscraper"),
    ("happy", "joyful"),
    ("happy", "sad"),
    ("king", "queen"),
    ("paris", "france"),
    ("tokyo", "japan"),
]

print(f"{'Word 1':<12} {'Word 2':<12} {'Similarity':>12}")
print("-" * 38)
for w1, w2 in word_pairs:
    sim = glove.similarity(w1, w2)
    print(f"{w1:<12} {w2:<12} {sim:>12.4f}")

In [ ]:
# Real vector arithmetic

print("=== Vector Arithmetic ===\n")

# king - man + woman
result = glove["king"] - glove["man"] + glove["woman"]
similar = glove.similar_by_vector(result, topn=5)
print("king - man + woman:")
for word, score in similar:
    print(f"  {word:<15} {score:.4f}")

print()

# paris - france + japan
result = glove["paris"] - glove["france"] + glove["japan"]
similar = glove.similar_by_vector(result, topn=5)
print("paris - france + japan (expect: tokyo):")
for word, score in similar:
    print(f"  {word:<15} {score:.4f}")

In [ ]:
# Find most similar words

print("=== Most Similar Words ===\n")

for word in ["dog", "computer", "music", "river"]:
    similar = glove.most_similar(word, topn=5)
    print(f"Most similar to '{word}':")
    for w, score in similar:
        print(f"  {w:<15} {score:.4f}")
    print()

---
## 7. The Fundamental Limitation of Static Embeddings

Word2Vec and GloVe give every word **one fixed vector** — forever, regardless of context.

This creates an unsolvable problem:

```
"I went to the river bank"      →  bank = [0.12, 0.55, ...]
"I went to the bank to deposit" →  bank = [0.12, 0.55, ...]
```

**Same vector. Different meaning.**

The word "bank" cannot be two things at once in a static embedding space. The model is forced to store some average of all its meanings, which is accurate for neither.

This is called the **polysemy problem** — a single word with multiple meanings gets stuck with one vector.

**The same problem applies to:**
- "bat" (animal vs. cricket bat)
- "light" (not heavy vs. illumination)
- "fly" (the insect vs. to fly)

To truly solve this, we need embeddings that **change based on context** — vectors that see the surrounding sentence and adjust accordingly.

That is what the **Attention Mechanism** solves — and it is the heart of the Transformer.

In [ ]:
# Demonstrate the polysemy problem

print("=== The Polysemy Problem ===\n")

# GloVe gives ONE vector for 'bank' regardless of context
bank_vector = glove["bank"]

# Find what 'bank' is most similar to
similar = glove.most_similar("bank", topn=10)
print("Words most similar to 'bank' in GloVe:")
for word, score in similar:
    print(f"  {word:<15} {score:.4f}")

print()
print("Notice: the vector is an AVERAGE of 'river bank' and 'financial bank'")
print("It captures neither meaning precisely.")
print()
print("To fix this, we need context-dependent embeddings.")
print("That is what the Transformer's Attention mechanism provides.")

---
## Quiz

Test your understanding before moving on.

**Q1.** What is the key difference between a Bag of Words vector and a word embedding vector?

**Q2.** Two words have a cosine similarity of 0.95. What does this tell you about them?

**Q3.** Word2Vec learns from raw text without any human labels. What is the training task it uses?

**Q4.** Why does a static embedding fail for the word "bank"?

**Q5.** If `vector("Paris") - vector("France") + vector("Germany") ≈ vector("Berlin")`, what relationship has the embedding space learned to encode?

**Q6.** In the Skip-gram model, what is the input and what is the model trying to predict?

In [ ]:
# Q1
answer_1 = """
Your answer here
"""

# Q2
answer_2 = """
Your answer here
"""

# Q3
answer_3 = """
Your answer here
"""

# Q4
answer_4 = """
Your answer here
"""

# Q5
answer_5 = """
Your answer here
"""

# Q6
answer_6 = """
Your answer here
"""

print("Answers recorded. Review with your instructor.")

---
## Summary

| Concept | What It Is | Limitation |
|---------|-----------|------------|
| Bag of Words | Word count vector | Loses word order |
| TF-IDF | Weighted count vector | Still no semantic meaning |
| Word Embedding | Dense learned vector | One fixed vector per word |
| Word2Vec | Learns embeddings from context prediction | Static — polysemy unsolved |
| GloVe | Learns from global co-occurrence statistics | Static — polysemy unsolved |

**Key takeaway:** Static embeddings capture meaning well, but fail when a word has multiple meanings depending on context. The solution — context-dependent embeddings computed via Attention — is the foundation of the Transformer, which we build toward in the next notebooks.

**Next:** Tokenization — how raw text becomes the token sequences that feed into embedding layers.